# Recherche par le sens : embeddings + base vectorielle

**Objectif** : découper les documents, calculer les embeddings, monter la base vectorielle, puis comparer recherche par mots-clés et recherche sémantique sur nos questions.

| Étape | Où |
|---|---|
| 1. Découper les PDF en passages (chunks) | `src/build_chunks.py` → `data/processed/chunks.jsonl` |
| 2. Calculer les embeddings + monter la base vectorielle | **ce notebook** |
| 3. Comparer mots-clés / sémantique sur les questions | `python src/compare_search.py` → `docs/evaluation/comparaison_recherche.md` |


## Préparation

En local : `pip install -r requirements.txt` une seule fois, depuis la racine du projet.

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "rag_config.py").exists()), None)
if ROOT is None:
    # Recherche locale dans le dossier extrait si le CWD ne permet pas de remonter directement
    config_paths = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        try:
            config_paths.extend(base.rglob("src/rag_config.py"))
        except PermissionError:
            pass

    if config_paths:
        ROOT = config_paths[0].parent.parent

    if ROOT is None:
        raise FileNotFoundError(
            "Extrayez puis ouvrez le notebook depuis le dossier COGNIVEX "
            "(C:\\...\\COGNIVEX\\notebooks\\recherche.ipynb)."
        )
sys.path.insert(0, str(ROOT / "src"))
print("Racine du projet :", ROOT)

from rag_config import (COLLECTION_NAME, EMBEDDING_MODEL, TOP_K, VECTOR_DB_DIR,
                        charger_chunks, passage_prefix, query_prefix, texte_indexe)
print("Modèle d'embeddings :", EMBEDDING_MODEL)

Racine du projet : c:\Users\lyly\Desktop\COGNIVEX
Modèle d'embeddings : intfloat/multilingual-e5-small


## 1. Charger les chunks

In [2]:
import json
from collections import Counter
from rag_config import (
    CHUNKS_FILE,
    MIN_CHUNK_CHARS,
    NETTOYER_BRUIT,
    charger_chunks,
    nettoyer_texte,
)

with CHUNKS_FILE.open(encoding="utf-8") as fichier:
    chunks_bruts = [json.loads(ligne) for ligne in fichier if ligne.strip()]

chunks = charger_chunks()

bruts_par_dci = Counter(chunk["dci"] for chunk in chunks_bruts)
gardes_par_dci = Counter(chunk["dci"] for chunk in chunks)

textes_modifies = 0
for chunk in chunks_bruts:
    texte = chunk["text"]
    texte_nettoye = nettoyer_texte(texte) if NETTOYER_BRUIT else texte

    if texte_nettoye != texte:
        textes_modifies += 1

print(f"Chunks avant nettoyage : {len(chunks_bruts)}")
print(f"Chunks conservés après nettoyage : {len(chunks)}")
print(f"Chunks éliminés : {len(chunks_bruts) - len(chunks)}")
print(f"Chunks dont le texte a été modifié : {textes_modifies}\n")

print("Détail par DCI :")
for dci in sorted(bruts_par_dci):
    avant = bruts_par_dci[dci]
    apres = gardes_par_dci[dci]
    elimines = avant - apres
    print(f"{dci:40s} avant : {avant:5d} | après : {apres:5d} | éliminés : {elimines:5d}")

Chunks avant nettoyage : 1367
Chunks conservés après nettoyage : 1367
Chunks éliminés : 0
Chunks dont le texte a été modifié : 0

Détail par DCI :
ACIDE-ACETYLSALICYLIQUE                  avant :    17 | après :    17 | éliminés :     0
AMLODIPINE                               avant :    55 | après :    55 | éliminés :     0
AMOXICILLINE                             avant :   184 | après :   184 | éliminés :     0
AMOXICILLINE + ACIDE CLAVULANIQUE        avant :    73 | après :    73 | éliminés :     0
ATÉNOLOL                                 avant :    25 | après :    25 | éliminés :     0
AZITHROMYCINE                            avant :    11 | après :    11 | éliminés :     0
BISOPROLOL                               avant :    19 | après :    19 | éliminés :     0
CEFTRIAXONE                              avant :    33 | après :    33 | éliminés :     0
CIPROFLOXACINE                           avant :    14 | après :    14 | éliminés :     0
CLARITHROMYCINE                          av

In [3]:
import csv
from rag_config import (
    EXPERIENCE,
    MAX_WORDS,
    OVERLAP_WORDS,
    EVAL_DIR,
)

longueurs = [len(chunk["text"].split()) for chunk in chunks]

if not longueurs:
    raise ValueError("Aucun chunk chargé pour cette expérience.")

fichier_resultats = EVAL_DIR / "comparaison_tailles_chunks.csv"
fichier_resultats.parent.mkdir(parents=True, exist_ok=True)

resultats = {}

if fichier_resultats.is_file():
    with fichier_resultats.open(encoding="utf-8-sig", newline="") as fichier:
        for ligne in csv.DictReader(fichier):
            resultats[ligne["experience"]] = ligne

resultats[str(EXPERIENCE)] = {
    "experience": str(EXPERIENCE),
    "max_mots": str(MAX_WORDS),
    "chevauchement": str(OVERLAP_WORDS),
    "nombre_chunks": str(len(chunks)),
    "nombre_dci": str(len({chunk["dci"] for chunk in chunks})),
    "taille_moyenne_reelle": f"{sum(longueurs) / len(longueurs):.1f}",
    "taille_min": str(min(longueurs)),
    "taille_max": str(max(longueurs)),
}

colonnes = [
    "experience",
    "max_mots",
    "chevauchement",
    "nombre_chunks",
    "nombre_dci",
    "taille_moyenne_reelle",
    "taille_min",
    "taille_max",
]

with fichier_resultats.open("w", encoding="utf-8-sig", newline="") as fichier:
    writer = csv.DictWriter(fichier, fieldnames=colonnes)
    writer.writeheader()
    writer.writerows(resultats[numero] for numero in sorted(resultats))

print(f"Expérience {EXPERIENCE} : {len(chunks)} chunks")
print(f"Taille moyenne réelle : {sum(longueurs) / len(longueurs):.1f} mots")
print(f"Tableau enregistré : {fichier_resultats}")

Expérience 3 : 1367 chunks
Taille moyenne réelle : 302.1 mots
Tableau enregistré : C:\Users\lyly\Desktop\COGNIVEX\docs\evaluation\comparaison_tailles_chunks.csv


## 2. Calculer les embeddings

Le modèle `multilingual-e5-small` comprend le français et accepte 512 tokens , le modèle (~470 Mo, internet nécessaire).


In [4]:
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer(EMBEDDING_MODEL)
print("Modèle chargé :", modele is not None)
print("Dimension :", modele.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modèle chargé : True
Dimension : 384


In [5]:
import numpy as np
import time
from rag_config import passage_prefix, texte_indexe, EMBEDDINGS_FILE

textes = [passage_prefix() + texte_indexe(chunk) for chunk in chunks]

debut = time.time()
embeddings = modele.encode(
    textes,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

EMBEDDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
np.save(EMBEDDINGS_FILE, embeddings)

print(f"{len(embeddings)} embeddings calculés en {time.time() - debut:.0f} secondes")
print("Dimension :", embeddings.shape[1])
print("Fichier enregistré :", EMBEDDINGS_FILE)

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

1367 embeddings calculés en 249 secondes
Dimension : 384
Fichier enregistré : C:\Users\lyly\Desktop\COGNIVEX\data\processed\embeddings\embeddings_exp3.npy


In [6]:
print("Norme du 1er vecteur :", round(float(np.linalg.norm(embeddings[0])), 4), "(1.0 = normalisé)")
print("8 premières valeurs  :", np.round(embeddings[0][:8], 3))

Norme du 1er vecteur : 1.0 (1.0 = normalisé)
8 premières valeurs  : [ 0.018 -0.031 -0.072 -0.054  0.031 -0.036  0.023  0.096]


## 3. Monter la base vectorielle (ChromaDB)

Pour éviter le crash du kernel pendant l’indexation, la base vectorielle ChromaDB a été construite à l’aide du script `src/build_vectordb.py`, exécuté séparément du notebook. La base générée est enregistrée dans `data/processed/vectordb/`. Le notebook peut ensuite la charger pour effectuer les recherches.

In [7]:
import chromadb

from rag_config import COLLECTION_NAME, VECTOR_DB_DIR

client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)

print("Vecteurs dans ChromaDB :", collection.count())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Vecteurs dans ChromaDB : 1367


## 4. Test rapide : recherche par le sens

`chercher()` transforme la question en vecteur (préfixe `query: `) et retourne les chunks les plus proches. Les questions viennent de `docs/evaluation/questions_llm.md`.

In [9]:
def chercher(question, k=TOP_K):
    emb = modele.encode([query_prefix() + question], normalize_embeddings=True)
    res = collection.query(query_embeddings=emb.tolist(), n_results=k)
    for rang, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
        print(f"{rang}. similarité {1 - dist:.3f} | {meta['dci']} | {meta['produit'] or '-'} | "
              f"{meta['doc_type']} p.{meta['page']}")
        print("   ", " ".join(doc.split())[:200], "\n")


from compare_search import charger_questions

for q in charger_questions()[:3]:          # changez [:3] pour tester d'autres questions
    print("=" * 100)
    print(q["id"], "—", q["texte"], "\n")
    chercher(q["texte"], k=3)

Q1 — Quelle est la liste des formes pharmaceutiques et des dosages de l’association amoxicilline + acide clavulanique répertoriés dans la nomenclature pharmaceutique algérienne consultée ? 

1. similarité 0.893 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | Notice p.12
    Amoxicilline.............................................................................................................. 100,00 mg Sous forme d'amoxicilline trihydratée Acide clavulanique........... 

2. similarité 0.892 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.16
    5.2. Propriétés pharmacocinétiques Absorption L’amoxicilline et l’acide clavulanique sont totalement dissociés en solution aqueuse à pH physiologique. Ces deux composants sont rapidement absorbés aprè 

3. similarité 0.891 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.17
    Les concentrations sériques d’amoxicilline et d’acide clavulanique 

## 5. Comparer avec la recherche par mots-clés
 
execute python src/compare_search.py

Le script pose **toutes les questions** de `docs/evaluation/questions_llm.md` aux deux méthodes (BM25 par mots-clés et recherche sémantique), sans réponse de référence, et écrit le rapport `docs/evaluation/comparaison_recherche_exp(n).md` (extraits retrouvés, médicaments retrouvés, chunks communs aux deux méthodes).

In [12]:
from openpyxl import load_workbook
from rag_config import ROOT, normalize, charger_chunks

excel = ROOT / "data" / "reference" / "Liste 40 medicaments final.xlsx"

workbook = load_workbook(excel, read_only=True, data_only=True)
sheet = workbook.active

headers = [
    str(cell).strip() if cell is not None else ""
    for cell in next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))
]

colonne_dci = "DCI (nom de la substance)"

if colonne_dci not in headers:
    raise ValueError(f"Colonne introuvable dans Excel : {colonne_dci}")

index_dci = headers.index(colonne_dci)

attendues = {}
for ligne in sheet.iter_rows(min_row=2, values_only=True):
    valeur = ligne[index_dci]
    if valeur:
        attendues[normalize(valeur)] = str(valeur).strip()

# Cette DCI n'a pas été retrouvée pendant la collecte
attendues.pop(normalize("hydrochlorothiazide"), None)

chunks = charger_chunks()
trouvees = {normalize(chunk["dci"]) for chunk in chunks}

print("DCI attendues :", len(attendues))
print("DCI présentes dans les chunks :", len(trouvees))
print("DCI manquantes :")
for cle in sorted(set(attendues) - trouvees):
    print("-", attendues[cle])

workbook.close()

DCI attendues : 39
DCI présentes dans les chunks : 36
DCI manquantes :
- cotrimoxazole (sulfaméthoxazole + triméthoprime)
- énalapril
- losartan
- ramipril


In [ ]:
from openpyxl import load_workbook
from rag_config import ROOT, normalize, charger_chunks

excel = ROOT / "data" / "reference" / "Liste 40 medicaments final.xlsx"

workbook = load_workbook(excel, read_only=True, data_only=True)
sheet = workbook.active

headers = [cell.value for cell in sheet[1]]
print("أعمدة Excel:", headers)

أعمدة Excel: ['N°', 'Classe', 'DCI (nom de la substance)', 'Code ATC (indicatif)', 'ATC confirmé', 'Présent dans la nomenclature algérienne ?', 'N,C', 'Forme', 'Dosage', 'Sources', 'Remarques', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [18]:
nom_colonne = "DCI (nom de la substance)"  # بدّليه بالاسم اللي ظهر في أعمدة Excel

indice = headers.index(nom_colonne)
dci_attendues = {
    str(row[indice]).strip()
    for row in sheet.iter_rows(min_row=2, values_only=True)
    if row[indice]
}

chunks = charger_chunks()
dci_chunks = {normalize(c["dci"]): c["dci"] for c in chunks}
dci_excel = {normalize(nom): nom for nom in dci_attendues}

print("DCI dans Excel et non présente dans chunks:")
for cle in sorted(dci_excel.keys() - dci_chunks.keys()):
    print("-", dci_excel[cle])

print("\nDCI dans chunks et non présente dans Excel:")
for cle in sorted(dci_chunks.keys() - dci_excel.keys()):
    print("-", dci_chunks[cle])

print(f"\nExcel: {len(dci_excel)} DCI présentes")
print(f"Chunks: {len(dci_chunks)} DCI présentes")

DCI dans Excel et non présente dans chunks:
- cotrimoxazole (sulfaméthoxazole + triméthoprime)
- énalapril
- hydrochlorothiazide
- losartan
- ramipril

DCI dans chunks et non présente dans Excel:
- COTRIMOXAZOLE

Excel: 40 DCI présentes
Chunks: 36 DCI présentes
